# 元素分布

统计训练集和生成结构中的元素分布并绘制热图。
运行前设置数据目录和输出目录。


In [ ]:
import csv
import os
import warnings
warnings.filterwarnings("ignore")
from pymatgen.core import Structure
from collections import Counter

In [ ]:
def get_element_frequencies_in_cifs(
        directory: str = "."
):
    """
    统计指定文件夹中所有 CIF 文件中元素的出现频率。

    Args:
        directory (str): 指定文件夹路径，默认为当前文件夹。

    Returns:
        dict: 元素频率字典，key 为元素符号 (str)，value 为出现频率百分比 (float)。
    """
    element_counter = Counter()
    total_elements = 0

    # 遍历文件夹中的所有 .cif 文件
    for file in os.listdir(directory):
        if file.endswith(".cif"):
            try:
                # 读取 CIF 文件的结构
                structure = Structure.from_file(os.path.join(directory, file))
                # 统计每种元素的数量
                for element in structure.composition:
                    count = structure.composition[element]
                    element_counter[element.symbol] += count
                    total_elements += count
            except Exception as e:
                print(f"Error processing {file}: {e}")

    # 计算元素频率百分比
    element_frequencies = {
        element: (count / total_elements * 100)
        for element, count in element_counter.items()
    }

    return element_frequencies

In [ ]:
work_dir = "C:/Users/13701/Desktop/gm_cifs/cdvae/v32/to_plot/"
dataset_name = "5sets_data"
plot_data_dir = os.path.join(work_dir, dataset_name)
plot_data = get_element_frequencies_in_cifs(plot_data_dir)
print(plot_data)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_element_heatmap(element_frequencies):
    """
    绘制元素频次热力图，将元素分布在周期表上。
    
    Args:
        element_frequencies (dict): 元素频率字典，key 为元素符号，value 为频率百分比 (float)。
    """
    # 元素周期表布局
    periodic_table_layout = [
        ["H",  None, None, None, None, None, None, None, None, None, None, None, None, "He"],
        ["Li", "Be", None, None, None, None, None, None, None, None, None, "B",  "C",  "N",  "O",  "F",  "Ne"],
        ["Na", "Mg", None, None, None, None, None, None, None, None, None, "Al", "Si", "P",  "S",  "Cl", "Ar"],
        ["K",  "Ca", "Sc", "Ti", "V",  "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As", "Se", "Br", "Kr"],
        ["Rb", "Sr", "Y",  "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "In", "Sn", "Sb", "Te", "I",  "Xe"],
        ["Cs", "Ba", None, "Hf", "Ta", "W",  "Re", "Os", "Ir", "Pt", "Au", "Hg", "Tl", "Pb", "Bi", "Po", "At", "Rn"],
        ["Fr", "Ra", None, "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds", "Rg", "Cn", "Nh", "Fl", "Mc", "Lv", "Ts", "Og"]
    ]

    # 计算实际布局的行数和最大列数
    num_rows = len(periodic_table_layout)
    num_cols = max(len(row) for row in periodic_table_layout)

    # 初始化热力图数据
    heatmap_data = np.zeros((num_rows, num_cols))
    element_positions = {}

    # 填充元素位置和频率
    for i, row in enumerate(periodic_table_layout):
        for j, element in enumerate(row):
            if element is not None:
                element_positions[element] = (i, j)
                heatmap_data[i, j] = element_frequencies.get(element, 0)

    # 创建热力图
    plt.figure(figsize=(15, 8))
    ax = sns.heatmap(
        heatmap_data,
        annot=False,
        cmap="YlGnBu",
        cbar_kws={"label": "Frequency (%)"},
        linewidths=0.5,
        linecolor="white",
        xticklabels=False,
        yticklabels=False,
    )

    # 在热力图上添加元素符号
    for element, (i, j) in element_positions.items():
        plt.text(j + 0.5, i + 0.5, element, ha="center", va="center", color="black")

    # 设置标题
    plt.title("Element Frequency Heatmap (Periodic Table Layout)", fontsize=16)
    plt.xlabel("")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_element_heatmap(element_frequencies=plot_data)

In [ ]:
from mendeleev import element
import matplotlib as mpl
import matplotlib.patches as patches


def plot_periodic_table_heatmap(plot_data):
    """
    绘制元素周期表热力图。

    Args:
        plot_data (dict): 元素数据字典，键为元素符号 (str)，值为热力值 (int/float)。
    """
    # 配置参数
    cell_length = 1
    cell_gap = 0.1
    cell_edge_width = 0.5

    # 准备元素数据
    elements = []
    for i in range(1, 119):
        ele = element(i)
        ele_group, ele_period = ele.group_id, ele.period

        # 特殊处理 La 和 Ac 系元素的分组与周期
        if 57 <= i <= 71:
            ele_group = i - 57 + 3
            ele_period = 8
        if 89 <= i <= 103:
            ele_group = i - 89 + 3
            ele_period = 9

        elements.append([i, ele.symbol, ele_group, ele_period, plot_data.get(ele.symbol, 0)])

    # 添加 La 和 Ac 系注解标签
    elements.append([None, 'LA', 3, 6, None])
    elements.append([None, 'AC', 3, 7, None])
    elements.append([None, 'LA', 2, 8, None])
    elements.append([None, 'AC', 2, 9, None])

    # 初始化绘图
    fig = plt.figure(figsize=(10, 5))
    xy_length = (20, 11)  # x, y 轴的范围

    # 获取颜色映射
    my_cmap = plt.cm.YlOrRd  # 修改为直接使用 plt.cm
    norm = mpl.colors.Normalize(1, 100)  # 根据实际情况调整上下限
    my_cmap.set_under('None')  # 设置低于范围的颜色（透明）
    cmmapable = mpl.cm.ScalarMappable(norm=norm, cmap=my_cmap)

    # 绘制颜色条
    plt.colorbar(cmmapable, drawedges=False)

    # 绘制元素周期表
    for e in elements:
        ele_number, ele_symbol, ele_group, ele_period, ele_count = e

        if ele_group is None:
            continue

        # 计算 cell 的位置
        x = (cell_length + cell_gap) * (ele_group - 1)
        y = xy_length[1] - ((cell_length + cell_gap) * ele_period)

        # 增加 La, Ac 系元素距离
        if ele_period >= 8:
            y -= cell_length * 0.5

        # 绘制 cell 和填充颜色
        if ele_number:
            fill_color = my_cmap(norm(ele_count))
            rect = patches.Rectangle(
                xy=(x, y), width=cell_length, height=cell_length,
                linewidth=cell_edge_width, edgecolor='k', facecolor=fill_color
            )
            plt.gca().add_patch(rect)

        # 添加原子序数
        plt.text(
            x + 0.04, y + 0.8, ele_number,
            va='center', ha='left',
            fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'}
        )

        # 添加元素符号
        plt.text(
            x + 0.5, y + 0.5, ele_symbol,
            va='center', ha='center',
            fontdict={'size': 9, 'color': 'black', 'family': 'Helvetica', 'weight': 'bold'}
        )

        # 添加热力值
        if ele_count is not None:
            plt.text(
                x + 0.5, y + 0.12, f"{ele_count:.1f}",
                va='center', ha='center',
                fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'}
            )

    # 设置图表属性
    plt.axis('equal')  # 保持单元格正方形
    plt.axis('off')  # 关闭坐标轴
    plt.tight_layout()  # 自动调整布局
    plt.ylim(0, xy_length[1])
    plt.xlim(0, xy_length[0])
    plt.show()

In [ ]:
plot_periodic_table_heatmap(plot_data)

In [ ]:
from matplotlib import cm
import mendeleev

cell_length = 1
# 各个cell的间隔
cell_gap = 0.1
# cell边框的粗细
cell_edge_width = 0.5

# 获取各个元素的原子序数、周期数（行数）、族数（列数）以及绘制数据（没有的设置为0）
elements = []
for i in range(1, 119):
    ele = mendeleev.element(i)
    ele_group, ele_period = ele.group_id, ele.period

    # 将La系元素设置到第8行
    if 57 <= i <= 71:
        ele_group = i - 57 + 3
        ele_period = 8
    # 将Ac系元素设置到第9行
    if 89 <= i <= 103:
        ele_group = i - 89 + 3
        ele_period = 9

    elements.append([i, ele.symbol, ele_group, ele_period,
                     plot_data.setdefault(ele.symbol, 0)])

# 设置La和Ac系的注解标签
elements.append([None, 'LA', 3, 6, None])
elements.append([None, 'AC', 3, 7, None])
elements.append([None, 'LA', 2, 8, None])
elements.append([None, 'AC', 2, 9, None])

# 新建Matplotlib绘图窗口
fig = plt.figure(figsize=(10, 5))
# x、y轴的范围
xy_length = (20, 11)

# 获取YlOrRd颜色条
# my_cmap = cm.get_cmap('YlOrRd')
my_cmap = plt.cm.YlOrRd
# 将plot_data数据映射为颜色，根据实际情况调整
norm = mpl.colors.Normalize(1, 100)
# 设置超出颜色条下界限的颜色（None为不设置，即白色）
my_cmap.set_under('None')
# 关联颜色条和映射
cmmapable = cm.ScalarMappable(norm=norm, cmap=my_cmap)
cmmapable.set_array([])
# 绘制颜色条
plt.colorbar(cmmapable, drawedges=False)

# 绘制元素周期表的cell，并填充属性和颜色
for e in elements:
    ele_number, ele_symbol, ele_group, ele_period, ele_count = e
    print(ele_number, ele_symbol, ele_group, ele_period, ele_count)

    if ele_group is None:
        continue

    # x, y定位cell的位置
    x = (cell_length + cell_gap) * (ele_group - 1)
    y = xy_length[1] - ((cell_length + cell_gap) * ele_period)

    # 增加 La, Ac 系元素距离元素周期表的距离
    if ele_period >= 8:
        y -= cell_length * 0.5

    # cell中原子序数部位None时绘制cell边框并填充热力颜色
    # 即不绘制La、Ac系注解标签地边框以及颜色填充
    if ele_number:
        fill_color = my_cmap(norm(ele_count))
        rect = patches.Rectangle(xy=(x, y),
                                 width=cell_length, height=cell_length,
                                 linewidth=cell_edge_width,
                                 edgecolor='k',
                                 facecolor=fill_color)
        plt.gca().add_patch(rect)

    # 在cell中添加原子序数属性
    plt.text(x + 0.04, y + 0.8,
             ele_number,
             va='center', ha='left',
             fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'})
    # 在cell中添加元素符号
    plt.text(x + 0.5, y + 0.5,
             ele_symbol,
             va='center', ha='center',
             fontdict={'size': 9, 'color': 'black', 'family': 'Helvetica', 'weight': 'bold'})
    # 在cell中添加热力值
    plt.text(x + 0.5, y + 0.12,
             ele_count,
             va='center', ha='center',
             fontdict={'size': 6, 'color': 'black', 'family': 'Helvetica'})

# x, y 轴设置等比例（1:1）（使cell看起来是正方形）
plt.axis('equal')
# 关闭坐标轴
plt.axis('off')
# 裁剪空白边缘
plt.tight_layout()
# 设置x, y轴的范围
plt.ylim(0, xy_length[1])
plt.xlim(0, xy_length[0])

# 将图保存为*.svg矢量格式
# plt.savefig('C:/Users/13701/Desktop/oxyf_periodic_table_percent.svg')
# 显示绘图窗口
plt.show()